# Клиенты торгового эквайринга

Один файл Excel, три листа. Строка — один ЦФТ ID. Колонки: ИНН и ЦФТ ID. ИНН из 9 цифр дополняется до 10, из 11 — до 12 и пишется текстом. Договор без ЦФТ ID в листы не входит.

Сбор идёт тремя отдельными запросами.

`С ТЭ на 01.09.2026` — договор SA и привязка `P` открыты в этот день.

`Без ТЭ на 01.09.2026` — организации `ods.scd1_z_client` (`class_id = 'CL_ORG'`, ИНН заполнен), которых нет на первом листе. У справочника нет даты, это текущий срез.

`Подключили ТЭ 06-08.2026` — ЦФТ ID, у которого хотя бы один договор SA открыт с 01.06.2026 по 31.08.2026, и привязка `P` открыта в день открытия этого договора. Договор, закрытый до 01.09.2026, на листе остаётся.

Смените даты и запустите ячейки по порядку. Файл ляжет в `/home/jovyan/documents/Equaring`.


In [ ]:
import getpass
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

AS_OF_DATE = '2026-09-01'
PERIOD_START = '2026-06-01'
PERIOD_END = '2026-08-31'
OUT_DIR = Path('/home/jovyan/documents/Equaring')

as_of = pd.to_datetime(AS_OF_DATE).strftime('%Y-%m-%d')
period_start = pd.to_datetime(PERIOD_START).strftime('%Y-%m-%d')
period_end = pd.to_datetime(PERIOD_END).strftime('%Y-%m-%d')

def ru_date(iso):
    year, month, day = iso.split('-')
    return '%s.%s.%s' % (day, month, year)

sheet_with = 'С ТЭ на %s' % ru_date(as_of)
sheet_without = 'Без ТЭ на %s' % ru_date(as_of)
sheet_opened = 'Подключили ТЭ %s-%s.%s' % (period_start[5:7], period_end[5:7], period_start[:4])
print(as_of, period_start, period_end)

LAKE_USER = input('Логин Impala: ').strip()
LAKE_PASSWORD = getpass.getpass('Пароль Impala: ')

imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': LAKE_USER, 'password': LAKE_PASSWORD},
)
imp._init_connection()
print('imp ready, user =', LAKE_USER)


In [ ]:
def as_clients(frame):
    out = pd.DataFrame({
        'ИНН': frame['inn'].map(lambda v: None if pd.isna(v) else str(v).strip()),
        'ЦФТ ID': frame['cft_id'].map(lambda v: None if pd.isna(v) else str(v).strip()),
    })
    out = out[out['ЦФТ ID'].notna() & out['ЦФТ ID'].ne('')]
    return out.drop_duplicates('ЦФТ ID').sort_values('ЦФТ ID').reset_index(drop=True)

sql_asof = f'''
WITH sa AS (
  SELECT DISTINCT
    a.n_agr,
    CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
  FROM ods_alpha.scd1_agreements a
  JOIN ods_alpha.scd1_companies c
    ON c.n_cmp = a.n_cmp_client
  WHERE upper(trim(a.acq_class)) = 'SA'
    AND a.d_valid_from <= '{as_of}'
    AND (a.d_valid_to IS NULL OR a.d_valid_to >= '{as_of}')
    AND coalesce(a.ods_deleted_flg, '0') <> '1'
    AND coalesce(c.ods_deleted_flg, '0') <> '1'
    AND c.c_inn IS NOT NULL
    AND EXISTS (
      SELECT 1
      FROM ods_alpha.scd1_agr_terms t
      WHERE t.n_agr = a.n_agr
        AND t.d_valid_from <= '{as_of}'
        AND (t.d_valid_to IS NULL OR t.d_valid_to >= '{as_of}')
        AND upper(trim(t.cf_ter_type)) = 'P'
        AND coalesce(t.ods_deleted_flg, '0') <> '1'
    )
),
ocrm AS (
  SELECT inn, row_id
  FROM (
    SELECT
      s.inn,
      soe.row_id,
      row_number() OVER (
        PARTITION BY s.inn
        ORDER BY soe.created DESC, soe.row_id DESC
      ) AS rn
    FROM (SELECT DISTINCT inn FROM sa) s
    JOIN ocrm_ul.s_org_ext soe
      ON CASE length(regexp_replace(soe.x_inn, '[^0-9]', ''))
           WHEN 9 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 10, '0')
           WHEN 11 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 12, '0')
           ELSE regexp_replace(soe.x_inn, '[^0-9]', '')
         END = s.inn
    WHERE soe.x_removed_flg = 'N'
      AND soe.x_duplicate_flg = 'N'
  ) t
  WHERE rn = 1
),
cdi AS (
  SELECT inn, cdi_id
  FROM (
    SELECT
      o.inn,
      e.party_id AS cdi_id,
      row_number() OVER (PARTITION BY o.inn ORDER BY e.party_id DESC) AS rn
    FROM ocrm o
    JOIN cdiul.ext_id_org e
      ON e.cmo_ext_party_source_id = o.row_id
     AND e.cmo_ext_source_system LIKE 'OCRM%'
  ) t
  WHERE rn = 1
),
cft AS (
  SELECT c.inn, e.cmo_ext_party_source_id AS cft_id
  FROM cdi c
  JOIN cdiul.ext_id_org e
    ON e.party_id = c.cdi_id
   AND e.cmo_ext_source_system LIKE 'CFT%'
)
SELECT cft_id, min(inn) AS inn
FROM (
  SELECT
    c.cft_id,
    s.inn,
    row_number() OVER (PARTITION BY s.inn ORDER BY c.cft_id) AS rn
  FROM sa s
  JOIN cft c ON c.inn = s.inn
) t
WHERE rn = 1
GROUP BY cft_id
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    with_te = as_clients(imp.fetch(sql_asof))
print(sheet_with, len(with_te))
display(with_te.head(20))


In [ ]:
sql_orgs = '''
SELECT cft_id, min(inn) AS inn
FROM (
  SELECT
    cast(id AS string) AS cft_id,
    CASE length(regexp_replace(c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c_inn, '[^0-9]', '')
    END AS inn
  FROM ods.scd1_z_client
  WHERE class_id = 'CL_ORG'
    AND c_inn IS NOT NULL
) t
GROUP BY cft_id
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    all_orgs = as_clients(imp.fetch(sql_orgs))
without_te = all_orgs.loc[~all_orgs['ЦФТ ID'].isin(set(with_te['ЦФТ ID']))].reset_index(drop=True)
print(sheet_without, len(without_te))
display(without_te.head(20))


In [ ]:
sql_opened = f'''
WITH sa AS (
  SELECT n_agr, inn
  FROM (
    SELECT
      a.n_agr,
      CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
    FROM ods_alpha.scd1_agreements a
    JOIN ods_alpha.scd1_companies c
      ON c.n_cmp = a.n_cmp_client
    WHERE upper(trim(a.acq_class)) = 'SA'
      AND a.d_valid_from >= '{period_start}'
      AND a.d_valid_from <= '{period_end}'
      AND coalesce(a.ods_deleted_flg, '0') <> '1'
      AND coalesce(c.ods_deleted_flg, '0') <> '1'
      AND c.c_inn IS NOT NULL
      AND EXISTS (
        SELECT 1
        FROM ods_alpha.scd1_agr_terms t
        WHERE t.n_agr = a.n_agr
          AND t.d_valid_from <= a.d_valid_from
          AND t.d_valid_to IS NULL
          AND upper(trim(t.cf_ter_type)) = 'P'
          AND coalesce(t.ods_deleted_flg, '0') <> '1'
      )
    UNION ALL
    SELECT
      a.n_agr,
      CASE length(regexp_replace(c.c_inn, '[^0-9]', ''))
      WHEN 9 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 10, '0')
      WHEN 11 THEN lpad(regexp_replace(c.c_inn, '[^0-9]', ''), 12, '0')
      ELSE regexp_replace(c.c_inn, '[^0-9]', '')
    END AS inn
    FROM ods_alpha.scd1_agreements a
    JOIN ods_alpha.scd1_companies c
      ON c.n_cmp = a.n_cmp_client
    WHERE upper(trim(a.acq_class)) = 'SA'
      AND a.d_valid_from >= '{period_start}'
      AND a.d_valid_from <= '{period_end}'
      AND coalesce(a.ods_deleted_flg, '0') <> '1'
      AND coalesce(c.ods_deleted_flg, '0') <> '1'
      AND c.c_inn IS NOT NULL
      AND EXISTS (
        SELECT 1
        FROM ods_alpha.scd1_agr_terms t
        WHERE t.n_agr = a.n_agr
          AND t.d_valid_from <= a.d_valid_from
          AND t.d_valid_to >= a.d_valid_from
          AND upper(trim(t.cf_ter_type)) = 'P'
          AND coalesce(t.ods_deleted_flg, '0') <> '1'
      )
  ) u
),
ocrm AS (
  SELECT inn, row_id
  FROM (
    SELECT
      s.inn,
      soe.row_id,
      row_number() OVER (
        PARTITION BY s.inn
        ORDER BY soe.created DESC, soe.row_id DESC
      ) AS rn
    FROM (SELECT DISTINCT inn FROM sa) s
    JOIN ocrm_ul.s_org_ext soe
      ON CASE length(regexp_replace(soe.x_inn, '[^0-9]', ''))
           WHEN 9 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 10, '0')
           WHEN 11 THEN lpad(regexp_replace(soe.x_inn, '[^0-9]', ''), 12, '0')
           ELSE regexp_replace(soe.x_inn, '[^0-9]', '')
         END = s.inn
    WHERE soe.x_removed_flg = 'N'
      AND soe.x_duplicate_flg = 'N'
  ) t
  WHERE rn = 1
),
cdi AS (
  SELECT inn, cdi_id
  FROM (
    SELECT
      o.inn,
      e.party_id AS cdi_id,
      row_number() OVER (PARTITION BY o.inn ORDER BY e.party_id DESC) AS rn
    FROM ocrm o
    JOIN cdiul.ext_id_org e
      ON e.cmo_ext_party_source_id = o.row_id
     AND e.cmo_ext_source_system LIKE 'OCRM%'
  ) t
  WHERE rn = 1
),
cft AS (
  SELECT c.inn, e.cmo_ext_party_source_id AS cft_id
  FROM cdi c
  JOIN cdiul.ext_id_org e
    ON e.party_id = c.cdi_id
   AND e.cmo_ext_source_system LIKE 'CFT%'
)
SELECT cft_id, min(inn) AS inn
FROM (
  SELECT
    c.cft_id,
    s.inn,
    row_number() OVER (PARTITION BY s.inn ORDER BY c.cft_id) AS rn
  FROM sa s
  JOIN cft c ON c.inn = s.inn
) t
WHERE rn = 1
GROUP BY cft_id
'''

with imp:
    imp.execute('set MEM_LIMIT=16g')
    opened = as_clients(imp.fetch(sql_opened))
print(sheet_opened, len(opened))
display(opened.head(20))


In [ ]:
out = OUT_DIR / ('trade_acquiring_clients_%s.xlsx' % as_of)
out.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(out, engine='openpyxl') as writer:
    with_te.to_excel(writer, index=False, sheet_name=sheet_with)
    without_te.to_excel(writer, index=False, sheet_name=sheet_without)
    opened.to_excel(writer, index=False, sheet_name=sheet_opened)
print(out)
